RAG PIPELINE - Data Ingestion to Vector DB Pipeline

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

d:\RAG Project\RAG-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    print(pdf_dir)

    # Find all pdfs Recursively
    pdf_files = list(pdf_dir.glob('**/*pdf'))

    print(f'Found length {len(pdf_files)} PDF files to process')
    
    for pdf_file in pdf_files:
            print(f'\n Processing {pdf_file.name}')
            try:                    
                loader = PyPDFLoader(str(pdf_file))
                documents = loader.load()
                # add source infomation to meta data
                for doc in documents:
                     doc.metadata['source_file'] = pdf_file.name,
                     doc.metadata['file_type'] = 'pdf'

                all_documents.extend(documents)
                print(f"  ✓ Loaded {len(documents)} pages")

            except Exception as e:
                 print(f'Error:{e}')

    return all_documents
                 

In [5]:
all_pdf_document = process_all_pdfs('../data/pdf')

..\data\pdf
Found length 4 PDF files to process

 Processing CEATLTD_28042026174742_BM_Outcome28042026signed.pdf
  ✓ Loaded 24 pages

 Processing HUDCO_28042026174840_SE_DISCLOSURE.pdf
  ✓ Loaded 3 pages

 Processing RATNAVEER_28042026175141_BM_Outcome_Final_signed.pdf
  ✓ Loaded 4 pages

 Processing SUPREMEINF_28042026175225_UPLOAD.pdf
  ✓ Loaded 4 pages


In [7]:
# Split documents (chunks)||

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [9]:
def split_in_chunks(doc,chunk_size=1000,chunk_overlap=200):
    text_splitter  = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
    )

    split_docs = text_splitter.split_documents(doc)

    print(f'Split {len(doc)} documents into {len(split_docs)} chunks')

    if split_docs:
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs


In [10]:
chunks = split_in_chunks(all_pdf_document)
print(chunks)

Split 35 documents into 120 chunks
Content: AP 
April 28, 2026 
  BSE Limited 
 Phiroze Jeejeebhoy Towers, 
 Dalal Street, 
 Mumbai 400 001 
 Security Code: 500878 
National Stock Exchange of India Limited 
Exchange Plaza, Bandra Kurla Complex,...
Metadata: {'producer': 'Adobe Acrobat (64-bit) 26 Paper Capture Plug-in', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-28T16:00:29+05:30', 'author': 'swati.sunki', 'company': 'Hewlett-Packard', 'contenttypeid': '0x01010057B6A642CBFA6C40803C96471D40D7FC', 'keywords': '', 'moddate': '2026-04-28T17:41:07+05:30', 'sourcemodified': '', 'subject': '', 'title': '', 'source': '..\\data\\pdf\\CEATLTD_28042026174742_BM_Outcome28042026signed.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1', 'source_file': ('CEATLTD_28042026174742_BM_Outcome28042026signed.pdf',), 'file_type': 'pdf'}
[Document(metadata={'producer': 'Adobe Acrobat (64-bit) 26 Paper Capture Plug-in', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '202

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple   
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class EmbeddingManager:
    def __init__(self,model = 'all-MiniLM-L6-v2'):
        print('model not loaded')
        self.model_name = model
        self.model = None
        self.load_model()

    def load_model(self):
        print(f'loading model')
        self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self,chunk:list[str]):
        if self.model is None:
            print(f'Model not laoded')
            
        embeddings = self.model.encode(chunk,show_progress_bar=True)
        print(embeddings)
        print(type(embeddings))
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


model not loaded
loading model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3267.80it/s]


In [28]:
## Vector Store

In [29]:
import os

In [47]:
class VectorStore:
    def __init__(self,path='../data/vector_store',collection_name='pdf_documents'):
        self.collection_name = collection_name
        self.path = path
        self.collection = None
        self.client = None
        self.initialize_store()

    def initialize_store(self):
        try:
            os.makedirs(self.path,exist_ok =True)
            self.client = chromadb.PersistentClient(path = self.path) # Like Create DB
            self.collection = self.client.get_or_create_collection( # Like Create Database
                name = self.collection_name,
                metadata={'description':'PDF document embeddings for RAG'}
                )
            print(f'vector store initialized.collection name {self.collection_name}')
            print(f'Existing documents in collection {self.collection.count()}')

        except Exception as e:
            print(f" Error intialized Vector Store: {e}")
            raise

    def add_documents(self,documents:list[Any],embeddings:np.ndarray):

        # id
        # documents
        # embeddings
        # metadata
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i,(doc,emb) in enumerate(zip(documents,embeddings)):
            doc_id = f'doc_{uuid.uuid4().hex[:8]}_{i}'
            ids.append(doc_id)

            print(type(doc))
            print(type(doc.metadata))
            
            # metadata = dict(doc.metadata)
            metadata = {k: (v[0] if isinstance(v, tuple) else v) 
                for k, v in doc.metadata.items()}
    
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                embeddings = embeddings_list,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total Documents added in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documnets to vector store:{e}")
            raise

In [48]:
vector_store = VectorStore()
vector_store

vector store initialized.collection name pdf_documents
Existing documents in collection 0


In [49]:
chunks

[Document(metadata={'producer': 'Adobe Acrobat (64-bit) 26 Paper Capture Plug-in', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-28T16:00:29+05:30', 'author': 'swati.sunki', 'company': 'Hewlett-Packard', 'contenttypeid': '0x01010057B6A642CBFA6C40803C96471D40D7FC', 'keywords': '', 'moddate': '2026-04-28T17:41:07+05:30', 'sourcemodified': '', 'subject': '', 'title': '', 'source': '..\\data\\pdf\\CEATLTD_28042026174742_BM_Outcome28042026signed.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1', 'source_file': ('CEATLTD_28042026174742_BM_Outcome28042026signed.pdf',), 'file_type': 'pdf'}, page_content='AP \nApril 28, 2026 \n  BSE Limited \n Phiroze Jeejeebhoy Towers, \n Dalal Street, \n Mumbai 400 001 \n Security Code: 500878 \nNational Stock Exchange of India Limited \nExchange Plaza, Bandra Kurla Complex,  \nBandra (East), Mumbai 400 051 \nSymbol: CEATLTD \nNCD Symbol: CL26, CL30 \nDear Sir/ Madam,  \nSub: Outcome of the Board Meeting held on April 28, 2026 \nPurs

In [50]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks,embeddings)

Batches: 100%|██████████| 4/4 [00:09<00:00,  2.26s/it]


[[-6.5812483e-02  2.2429962e-02 -2.4527244e-02 ... -9.6816584e-02
  -5.3851157e-02 -2.3827191e-02]
 [-6.6994280e-02 -3.5326432e-02 -2.8527489e-02 ... -8.4462382e-02
  -4.2572942e-02  4.7768094e-02]
 [-2.2604689e-02  4.7854153e-03 -7.5159505e-02 ... -7.8822553e-02
   4.5877947e-03  3.8648713e-02]
 ...
 [ 6.8636429e-03  6.6896761e-03 -1.6832810e-02 ... -1.3997422e-02
  -2.0825857e-02 -1.4583170e-02]
 [-6.3989580e-02  7.2017283e-05 -4.2891778e-02 ... -7.0268649e-04
  -5.0758487e-03 -3.9705329e-02]
 [-9.3961298e-02 -2.9971631e-02 -5.9229001e-02 ... -6.5127604e-02
  -1.9548204e-02  2.6422907e-02]]
<class 'numpy.ndarray'>
Generated embeddings with shape: (120, 384)
<class 'langchain_core.documents.base.Document'>
<class 'dict'>
<class 'langchain_core.documents.base.Document'>
<class 'dict'>
<class 'langchain_core.documents.base.Document'>
<class 'dict'>
<class 'langchain_core.documents.base.Document'>
<class 'dict'>
<class 'langchain_core.documents.base.Document'>
<class 'dict'>
<class 'lang